<a href="https://colab.research.google.com/github/edwardoughton/IGARSS26/blob/main/notebook_2_congo_river_water_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ IGARSS Summer School Exercise
## Mapping persistent water channels on the Congo River with Sentinel-2 and STAC

**Scenario.** A fiber-optic cable is being considered along a river route in the Democratic Republic of the Congo (DRC). During the dry season, a cable laid in a shallow or seasonally exposed part of the channel could be vulnerable to anchors, propellers, passing boats, erosion, and direct human activity.

Your task is to use four low-cloud Sentinel-2 observations, one from each year from 2022 to 2025, to estimate, for every river pixel, the **fraction of valid observations classified as water**. The resulting map will help a field-survey boat prioritize areas that appear persistently inundated while keeping the workflow fast enough for class.

> **Important engineering limitation:** Sentinel-2 optical imagery and NDWI do **not** measure water depth. A high water-occurrence probability is a screening proxy for persistent inundation, not proof of a deep or safe cable corridor. Final routing requires bathymetry/sonar, hydrology, navigation, geotechnical investigation, permitting, and local knowledge.

**Suggested duration:** 60-90 minutes  
**Study area:** Congo-Kasai river corridor near Kwamouth, DRC  
**Primary data source:** Sentinel-2 Level-2A surface reflectance from the Microsoft Planetary Computer STAC API


## Learning objectives

By the end of this exercise, you should be able to:

1. Search a STAC catalog by place, date, collection, and cloud cover.
2. Select one low-cloud Sentinel-2 scene per year for a fast classroom workflow.
3. Stream Sentinel-2 bands and the Scene Classification Layer (SCL) into an `xarray` data cube.
4. Clip analysis pixels to a river-corridor geometry.
5. Mask clouds, shadows, snow/ice, and invalid pixels.
6. Calculate NDWI and classify water for multiple dry-season years.
7. Distinguish **water occurrence**, **observation count**, and **classification uncertainty**.
8. Export analysis-ready GeoTIFFs and a table of candidate survey targets.
9. Explain why persistent surface water is not equivalent to deep water.


## Exercise deliverables

Submit:

- a map of valid-observation count;
- a map of dry-season water-occurrence probability;
- a map of mean dry-season NDWI;
- a GeoTIFF containing water probability;
- a short ranked table of candidate field-survey zones;
- a 200–300 word interpretation covering uncertainty and recommended field checks.

## 1. Install dependencies

This notebook follows the Python-oriented workflow used in the preceding data-acquisition notebook. It uses `pystac-client`, `planetary-computer`, `odc-stac`, `xarray`, `rioxarray`, `rasterio`, `GeoPandas`, and `Matplotlib`.

In [ ]:
# Run once in Google Colab. Restarting the runtime is normally unnecessary.
!pip -q install pystac-client planetary-computer odc-stac rioxarray rasterio geopandas shapely pyogrio folium mapclassify scipy

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import pystac_client
import planetary_computer
import odc.stac

import rioxarray  # activates the .rio accessor
import rasterio
from rasterio.features import geometry_mask, shapes

import geopandas as gpd
from shapely.geometry import Polygon, box, shape
from scipy import ndimage

warnings.filterwarnings("ignore")

DATA_DIR = Path("data/congo_water_probability")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Outputs will be written to:", DATA_DIR.resolve())


## 2. Define the study area, analysis years, and river clip

The default bounding box covers a manageable section of the Congo-Kasai river system near Kwamouth. It is intentionally small enough for a classroom workflow.

The example uses July as a practical dry-season screening window for western DRC. For engineering work, confirm the operational low-water season using gauge records, local hydrological expertise, and the precise route location.

The river clip polygon below is deliberately separate from the rectangular STAC search box. The box finds imagery; the polygon decides which pixels are allowed into the water-occurrence calculation. If you change the AOI, redraw the polygon as well.


In [ ]:
# WGS84 bounding box: [min longitude, min latitude, max longitude, max latitude]
AOI_BBOX = [16.05, -3.38, 16.55, -2.93]
AOI_NAME = "Congo-Kasai corridor near Kwamouth, DRC"

# Four classroom-friendly dry-season years: one selected image per year.
YEARS = list(range(2022, 2026))
START_MONTH_DAY = "07-01"
END_MONTH_DAY = "07-31"
SCENES_PER_YEAR = 1

# Scene-level filter. Pixel-level SCL masking is still essential.
MAX_SCENE_CLOUD = 10

# NDWI threshold to test. There is no universally correct threshold.
NDWI_THRESHOLD = 0.10

# Require enough valid observations before interpreting probability.
MIN_VALID_OBSERVATIONS = 2

# Approximate river-corridor polygon for the default AOI.
# Coordinates are (longitude, latitude) in EPSG:4326.
RIVER_CLIP_POLYGON_WGS84 = [
    (16.065, -3.365),
    (16.095, -3.285),
    (16.165, -3.185),
    (16.245, -3.075),
    (16.340, -2.955),
    (16.535, -2.945),
    (16.515, -3.025),
    (16.420, -3.115),
    (16.340, -3.190),
    (16.270, -3.285),
    (16.230, -3.375),
    (16.065, -3.365),
]

river_clip_gdf_wgs84 = gpd.GeoDataFrame(
    {"name": ["river_clip"]},
    geometry=[Polygon(RIVER_CLIP_POLYGON_WGS84)],
    crs="EPSG:4326",
)

print(AOI_NAME)
print("Bounding box:", AOI_BBOX)
print("Years:", YEARS)
print("Scenes per year:", SCENES_PER_YEAR)
print("River clip bounds:", river_clip_gdf_wgs84.total_bounds)


In [ ]:
# Optional: inspect the AOI and river clip on a simple interactive map.
import folium

center = [(AOI_BBOX[1] + AOI_BBOX[3]) / 2,
          (AOI_BBOX[0] + AOI_BBOX[2]) / 2]

m = folium.Map(location=center, zoom_start=10, tiles="CartoDB positron")
folium.Rectangle(
    bounds=[[AOI_BBOX[1], AOI_BBOX[0]], [AOI_BBOX[3], AOI_BBOX[2]]],
    tooltip="STAC search box",
    color="red",
    fill=False,
).add_to(m)
folium.GeoJson(
    river_clip_gdf_wgs84.__geo_interface__,
    name="River clip",
    tooltip=AOI_NAME,
    style_function=lambda feature: {
        "color": "blue",
        "weight": 2,
        "fillColor": "blue",
        "fillOpacity": 0.08,
    },
).add_to(m)
folium.LayerControl().add_to(m)
m


### ✏️ Exercise 1 - Choose a river reach

Change `AOI_BBOX` to another section of the Congo River or one of the proposed river-cable corridors. Keep the first run small (roughly 20-50 km across), and redraw `RIVER_CLIP_POLYGON_WGS84` so the analysis is clipped to the river corridor rather than the full rectangular search box.

Record:

- the approximate reach name;
- why it is operationally relevant;
- whether islands, sandbars, tributary confluences, settlements, or navigation channels are visible.


## 3. Connect to the STAC catalog and search Sentinel-2

We use the `sentinel-2-l2a` collection. The required assets are:

- `B03`: green reflectance at 10 m;
- `B08`: near-infrared reflectance at 10 m;
- `SCL`: Scene Classification Layer at 20 m, resampled to the analysis grid.

The search looks through the July window for each year and keeps the lowest-cloud scene for that year. This gives exactly four images for the default 2022-2025 classroom run.

NDWI is:

\[
NDWI = \frac{Green - NIR}{Green + NIR}
\]

Open water commonly has positive NDWI, but thresholds vary with turbidity, sun glint, shadows, floating vegetation, wetlands, sediment, and atmospheric conditions.


In [ ]:
STAC_API_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

catalog = pystac_client.Client.open(
    STAC_API_URL,
    modifier=planetary_computer.sign_inplace,
)

print("Connected to:", catalog.title)

In [ ]:
date_ranges = {
    year: f"{year}-{START_MONTH_DAY}/{year}-{END_MONTH_DAY}"
    for year in YEARS
}

all_items = []
candidate_records = []

for year, date_range in date_ranges.items():
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=AOI_BBOX,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": MAX_SCENE_CLOUD}},
    )
    year_items = sorted(
        list(search.items()),
        key=lambda item: (
            item.properties.get("eo:cloud_cover", np.inf),
            item.datetime,
        ),
    )

    if not year_items:
        raise RuntimeError(
            f"No scenes found for {date_range}. Enlarge the date range, "
            "relax MAX_SCENE_CLOUD, or check the AOI coordinates."
        )

    selected_items = year_items[:SCENES_PER_YEAR]
    selected_ids = {item.id for item in selected_items}

    for item in year_items:
        candidate_records.append(
            {
                "year": year,
                "item_id": item.id,
                "datetime": item.datetime,
                "cloud_cover": item.properties.get("eo:cloud_cover", np.nan),
                "mgrs_tile": item.properties.get("s2:mgrs_tile", ""),
                "selected_for_loading": item.id in selected_ids,
            }
        )

    print(
        f"{date_range}: selected {len(selected_items)} of {len(year_items)} "
        f"candidate scenes; best cloud cover = "
        f"{selected_items[0].properties.get('eo:cloud_cover', np.nan):.2f}%"
    )
    all_items.extend(selected_items)

# Remove duplicate STAC items and sort by time.
items_by_id = {item.id: item for item in all_items}
items = sorted(items_by_id.values(), key=lambda item: item.datetime)

print("\nUnique scenes selected for loading:", len(items))
if len(items) != len(YEARS) * SCENES_PER_YEAR:
    print("Warning: duplicate scene IDs reduced the final image count.")
for item in items:
    print(item.id, item.datetime, item.properties.get("eo:cloud_cover", np.nan))


In [ ]:
# Create a full scene inventory and a compact selected-scene inventory.
scene_inventory = pd.DataFrame(candidate_records).sort_values(
    ["year", "selected_for_loading", "cloud_cover"],
    ascending=[True, False, True],
)
scene_inventory["date"] = pd.to_datetime(scene_inventory["datetime"]).dt.date
scene_inventory.to_csv(DATA_DIR / "sentinel2_scene_inventory.csv", index=False)

selected_scene_inventory = scene_inventory.loc[
    scene_inventory["selected_for_loading"]
].sort_values("datetime")
selected_scene_inventory.to_csv(
    DATA_DIR / "sentinel2_selected_scene_inventory.csv",
    index=False,
)

display(selected_scene_inventory)
print("Selected scenes:", len(selected_scene_inventory))
print("Median selected catalog cloud cover:", selected_scene_inventory["cloud_cover"].median())


## 4. Load an analysis-ready data cube

`odc-stac` aligns scenes to one grid. We request a 10 m output grid and group acquisitions by solar day.

The default workflow loads only four scenes, so it should run quickly in class. The river clip is rasterized onto the same grid immediately after loading.


In [ ]:
ds = odc.stac.load(
    items,
    bands=["B03", "B08", "SCL"],
    bbox=AOI_BBOX,
    resolution=10,
    groupby="solar_day",
    chunks={"time": 1, "x": 2048, "y": 2048},
)

river_clip_gdf = river_clip_gdf_wgs84.to_crs(str(ds.odc.crs))
river_mask_values = geometry_mask(
    river_clip_gdf.geometry,
    out_shape=(ds.sizes["y"], ds.sizes["x"]),
    transform=ds.odc.geobox.transform,
    invert=True,
)
river_mask = xr.DataArray(
    river_mask_values,
    coords={"y": ds.coords["y"], "x": ds.coords["x"]},
    dims=("y", "x"),
    name="river_clip_mask",
)

print(ds)
print("Time steps:", ds.sizes["time"])
print("Raster shape:", ds.sizes["y"], "rows x", ds.sizes["x"], "columns")
print("CRS:", ds.odc.crs)
print("River-clip pixels:", int(river_mask.sum().item()))


## 5. Build a pixel-level quality mask

Sentinel-2's SCL codes include vegetation, bare soil, water, cloud shadow, clouds, cirrus, and snow/ice. We mask values that should not be used for NDWI:

| SCL | Meaning | Use? |
|---:|---|:---:|
| 0 | No data | No |
| 1 | Saturated/defective | No |
| 2 | Dark-area pixels | No |
| 3 | Cloud shadow | No |
| 4 | Vegetation | Yes |
| 5 | Bare soil | Yes |
| 6 | Water | Yes |
| 7 | Unclassified | Yes |
| 8 | Cloud, medium probability | No |
| 9 | Cloud, high probability | No |
| 10 | Thin cirrus | No |
| 11 | Snow/ice | No |

The final `valid` mask also clips pixels to the river-corridor polygon. Pixels outside that polygon are excluded from every downstream map and export.


In [ ]:
INVALID_SCL = [0, 1, 2, 3, 8, 9, 10, 11]

quality_valid = ~ds["SCL"].isin(INVALID_SCL)
valid = quality_valid & river_mask

# Sentinel-2 L2A reflectance is stored as scaled integers.
green = (ds["B03"].astype("float32") / 10000.0).where(valid)
nir = (ds["B08"].astype("float32") / 10000.0).where(valid)

# Remove impossible/out-of-range values.
physical = (
    (green >= 0) & (green <= 1.2) &
    (nir >= 0) & (nir <= 1.2)
)
green = green.where(physical)
nir = nir.where(physical)

valid = (green.notnull() & nir.notnull()) & river_mask
valid


## 6. Compute NDWI and classify water

The classification below combines an NDWI threshold with a weak NIR reflectance constraint. The NIR constraint helps reduce false positives from some bright or mixed surfaces, but it must be tested rather than accepted blindly.

A useful class discussion is to compare:

1. `NDWI > 0.00`;
2. `NDWI > 0.10`;
3. `NDWI > 0.20`;
4. SCL class 6 alone;
5. NDWI plus a second index such as MNDWI, which requires SWIR.

In [ ]:
def normalized_difference(a, b):
    denominator = a + b
    return xr.where(np.abs(denominator) > 1e-6, (a - b) / denominator, np.nan)

ndwi = normalized_difference(green, nir).astype("float32").where(river_mask)

# Adjustable screening rule.
water = ((ndwi > NDWI_THRESHOLD) & (nir < 0.25)).where(valid).where(river_mask)

print(ndwi)


In [ ]:
# Quick-look: one observation.
i = 0

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

green.isel(time=i).plot.imshow(ax=axes[0], cmap="gray", robust=True)
axes[0].set_title(f"Green reflectance\n{pd.to_datetime(ds.time.values[i]).date()}")

ndwi.isel(time=i).plot.imshow(
    ax=axes[1], cmap="BrBG", vmin=-0.5, vmax=0.5
)
axes[1].set_title("NDWI clipped to river")

water.isel(time=i).plot.imshow(
    ax=axes[2], cmap="Blues", vmin=0, vmax=1, add_colorbar=False
)
axes[2].set_title(f"Water: NDWI > {NDWI_THRESHOLD}")

for ax in axes:
    river_clip_gdf.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()


### ✏️ Exercise 2 — Threshold sensitivity

Run the classification with at least three NDWI thresholds. For each threshold:

- inspect a wide channel, island edge, sandbar, cloud edge, and settlement;
- note likely omission errors (water classified as land);
- note likely commission errors (land/shadow classified as water);
- choose and justify a threshold for the final probability map.

## 7. Estimate water-occurrence probability

For river pixel \(p\):

\[
P(\mathrm{water}_p) =
\frac{\text{number of valid selected observations classified as water}}
{\text{number of valid selected observations}}
\]

With the default four-image workflow, possible probabilities are coarse: 0.00, 0.25, 0.50, 0.75, and 1.00 when all four observations are valid. The denominator is critical. A pixel observed only once should not be treated with the same confidence as a pixel observed three or four times.


In [ ]:
valid_count = valid.sum(dim="time").astype("uint16").where(river_mask, 0)
water_count = water.fillna(False).sum(dim="time").astype("uint16").where(river_mask, 0)
water_probability = (
    water_count / valid_count.where(valid_count > 0)
).astype("float32").where(river_mask)
mean_ndwi = ndwi.mean(dim="time", skipna=True).astype("float32").where(river_mask)
ndwi_std = ndwi.std(dim="time", skipna=True).astype("float32").where(river_mask)

# Mask probability where there are too few valid observations.
water_probability_reliable = water_probability.where(
    (valid_count >= MIN_VALID_OBSERVATIONS) & river_mask
)

summary_ds = xr.Dataset(
    {
        "river_clip_mask": river_mask,
        "valid_observation_count": valid_count,
        "water_observation_count": water_count,
        "water_probability": water_probability,
        "water_probability_reliable": water_probability_reliable,
        "mean_ndwi": mean_ndwi,
        "ndwi_std": ndwi_std,
    }
)

summary_ds


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

summary_ds["valid_observation_count"].plot.imshow(
    ax=axes[0, 0], cmap="viridis"
)
axes[0, 0].set_title("Valid Sentinel-2 observations")

summary_ds["water_probability_reliable"].plot.imshow(
    ax=axes[0, 1], cmap="Blues", vmin=0, vmax=1
)
axes[0, 1].set_title(
    f"Dry-season water probability\n(minimum {MIN_VALID_OBSERVATIONS} observations)"
)

summary_ds["mean_ndwi"].plot.imshow(
    ax=axes[1, 0], cmap="BrBG", vmin=-0.4, vmax=0.5
)
axes[1, 0].set_title("Mean dry-season NDWI")

summary_ds["ndwi_std"].plot.imshow(
    ax=axes[1, 1], cmap="magma", vmin=0, vmax=0.25
)
axes[1, 1].set_title("NDWI temporal standard deviation")

for ax in axes.ravel():
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

### Reading the four maps

- **High probability + enough observations + low NDWI variability:** strongest remote-sensing evidence for persistent surface water inside the river clip.
- **High probability + few observations:** potentially useful, but confidence is low.
- **Intermediate probability:** seasonally exposed bars, moving banks, mixed pixels, floating vegetation, turbidity effects, or classification error.
- **High variability:** changing waterline, channel migration, clouds/shadows, or an unstable classifier.


## 8. Compare annual dry-season classifications

A single pooled probability can hide year-to-year changes. Because the classroom workflow loads one selected image per year, each panel below is an annual water classification for the lowest-cloud July scene that year.


In [ ]:
water_by_year = water.groupby("time.year").mean(dim="time", skipna=True)
valid_by_year = valid.groupby("time.year").sum(dim="time")

# With one selected scene per year, one valid observation is enough to show that year.
water_by_year = water_by_year.where(valid_by_year >= 1).where(river_mask)

n_years = water_by_year.sizes["year"]
ncols = 2
nrows = int(np.ceil(n_years / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 5 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, year in zip(axes, water_by_year.year.values):
    water_by_year.sel(year=year).plot.imshow(
        ax=ax, cmap="Blues", vmin=0, vmax=1, add_colorbar=False
    )
    river_clip_gdf.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(f"Selected-scene water classification: {int(year)}")
    ax.set_aspect("equal")

for ax in axes[n_years:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


### ✏️ Exercise 3 — Temporal interpretation

Identify at least two areas where the apparent channel changes between years. For each area, propose competing explanations:

- real river-stage change;
- bar emergence or erosion;
- channel migration;
- cloud/shadow contamination;
- threshold sensitivity;
- mixed pixels at the bank;
- aquatic or floating vegetation;
- geolocation/resampling effects.

## 9. Export analysis products

The helper below writes cloud-optimized, tiled GeoTIFF-compatible outputs using the georeferencing carried by the `odc-stac` cube.

In [ ]:
def write_geotiff(data_array, path, dtype="float32", nodata=-9999):
    da = data_array.compute()
    da = da.rio.write_crs(ds.odc.crs, inplace=False)
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

    if np.issubdtype(np.dtype(dtype), np.floating):
        encoded = da.fillna(nodata).astype(dtype)
    else:
        encoded = da.fillna(0).astype(dtype)

    encoded.rio.to_raster(
        path,
        driver="GTiff",
        compress="DEFLATE",
        tiled=True,
        nodata=nodata if np.issubdtype(np.dtype(dtype), np.floating) else 0,
    )
    print("Wrote:", path)

write_geotiff(
    summary_ds["water_probability_reliable"],
    DATA_DIR / "dry_season_water_probability.tif",
)
write_geotiff(
    summary_ds["valid_observation_count"],
    DATA_DIR / "valid_observation_count.tif",
    dtype="uint16",
    nodata=0,
)
write_geotiff(
    summary_ds["mean_ndwi"],
    DATA_DIR / "mean_dry_season_ndwi.tif",
)
write_geotiff(
    summary_ds["ndwi_std"],
    DATA_DIR / "ndwi_temporal_std.tif",
)

In [ ]:
# Store the compact summary cube for reproducibility.
summary_ds.compute().to_netcdf(DATA_DIR / "congo_water_summary.nc")

metadata = {
    "aoi_name": AOI_NAME,
    "aoi_bbox_wgs84": AOI_BBOX,
    "river_clip_polygon_wgs84": RIVER_CLIP_POLYGON_WGS84,
    "years": YEARS,
    "season_start": START_MONTH_DAY,
    "season_end": END_MONTH_DAY,
    "scenes_per_year": SCENES_PER_YEAR,
    "selected_item_ids": [item.id for item in items],
    "max_scene_cloud": MAX_SCENE_CLOUD,
    "ndwi_threshold": NDWI_THRESHOLD,
    "nir_constraint": "< 0.25",
    "minimum_valid_observations": MIN_VALID_OBSERVATIONS,
    "stac_api": STAC_API_URL,
    "collection": "sentinel-2-l2a",
    "warning": "Persistent surface-water proxy only; not a bathymetric depth product.",
}

with open(DATA_DIR / "analysis_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved NetCDF and metadata.")


## 10. Derive candidate survey zones

For classroom purposes, define a **persistent-water screening mask** using:

- water probability >= 0.90;
- enough valid observations across the four selected years;
- mean NDWI >= 0.15;
- low-to-moderate NDWI variability;
- the river clip mask.

Then remove tiny isolated patches and convert the remaining raster regions to polygons.

These zones are **not cable routes**. They are targets where a survey boat could prioritize sonar transects, current measurements, sediment sampling, anchor-risk observations, and navigation interviews.


In [ ]:
PERSISTENCE_THRESHOLD = 0.90
MEAN_NDWI_THRESHOLD = 0.15
MAX_NDWI_STD = 0.18
MIN_CANDIDATE_VALID_OBSERVATIONS = min(len(YEARS), 3)
MIN_PATCH_PIXELS = 100  # 100 pixels at 10 m is approximately 1 hectare

candidate = (
    (summary_ds["water_probability_reliable"] >= PERSISTENCE_THRESHOLD) &
    (summary_ds["valid_observation_count"] >= MIN_CANDIDATE_VALID_OBSERVATIONS) &
    (summary_ds["mean_ndwi"] >= MEAN_NDWI_THRESHOLD) &
    (summary_ds["ndwi_std"] <= MAX_NDWI_STD) &
    summary_ds["river_clip_mask"]
).compute()

# Remove small connected components.
labels, n_labels = ndimage.label(candidate.values.astype(bool))
component_sizes = np.bincount(labels.ravel())
keep_labels = np.where(component_sizes >= MIN_PATCH_PIXELS)[0]
keep_labels = keep_labels[keep_labels != 0]

clean_candidate = np.isin(labels, keep_labels) & river_mask.values
print("Connected components before filtering:", n_labels)
print("Components retained:", len(keep_labels))


In [ ]:
# Convert retained raster patches to polygons.
transform = ds.odc.geobox.transform
crs = ds.odc.crs

records = []
for geom, value in shapes(
    clean_candidate.astype("uint8"),
    mask=clean_candidate,
    transform=transform,
):
    if value == 1:
        records.append({"geometry": shape(geom)})

candidate_gdf = gpd.GeoDataFrame(records, crs=crs)

if len(candidate_gdf):
    # Calculate area and a simple ranking score.
    candidate_gdf["area_km2"] = candidate_gdf.area / 1_000_000
    candidate_gdf = candidate_gdf.sort_values("area_km2", ascending=False).reset_index(drop=True)
    candidate_gdf["survey_priority"] = np.arange(1, len(candidate_gdf) + 1)

    # Save in projected CRS and WGS84.
    candidate_gdf.to_file(DATA_DIR / "candidate_survey_zones.gpkg", driver="GPKG")
    candidate_wgs84 = candidate_gdf.to_crs(4326)
    candidate_wgs84.to_file(DATA_DIR / "candidate_survey_zones.geojson", driver="GeoJSON")

    display(candidate_gdf[["survey_priority", "area_km2"]].head(10))
else:
    print(
        "No candidate zones met all criteria. Relax one threshold at a time "
        "and document the effect."
    )

In [ ]:
# Plot candidate zones over water probability.
fig, ax = plt.subplots(figsize=(12, 10))

summary_ds["water_probability_reliable"].plot.imshow(
    ax=ax, cmap="Blues", vmin=0, vmax=1
)
river_clip_gdf.boundary.plot(ax=ax, color="black", linewidth=1.0)

if len(candidate_gdf):
    candidate_gdf.boundary.plot(ax=ax, linewidth=1.2, color="yellow")

ax.set_title(
    "Candidate field-survey zones\n"
    "(persistent surface-water proxy, not inferred depth)"
)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


### ✏️ Exercise 4 — Design the boat survey

For the three highest-priority candidate zones, propose a field plan. Include:

- sonar/bathymetric transects across and along the channel;
- water-level or gauge reference;
- current velocity and seasonal-stage information;
- sediment/substrate observations;
- navigational traffic, anchor use, fishing activity, and local landing sites;
- bank stability and evidence of erosion or deposition;
- potential cable burial/protection requirements;
- environmental and community constraints.

Explain how the satellite product reduces field effort without replacing the survey.

## 11. Optional extension — Add MNDWI

MNDWI often separates water from built-up or bare surfaces better than NDWI:

\[
MNDWI = \frac{Green - SWIR}{Green + SWIR}
\]

Add Sentinel-2 band `B11` to the STAC load, resample it to 10 m, calculate MNDWI, and compare classifications. Because B11 is natively 20 m, discuss whether a 10 m output grid creates genuinely finer information.

In [ ]:
# Optional extension starter:
#
# 1. Add "B11" to bands in the odc.stac.load() call.
# 2. swir = (ds["B11"].astype("float32") / 10000.0).where(valid)
# 3. mndwi = normalized_difference(green, swir)
# 4. Compare NDWI-only, MNDWI-only, and combined classifications.
#
# Your code here:

## 12. Interpretation questions

Answer in a markdown cell:

1. What exactly does a pixel value of 0.85 mean on your probability map?
2. Why is the valid-observation count map required?
3. How could the selected months bias the result?
4. How do turbidity and suspended sediment affect green and NIR reflectance?
5. Why might floating vegetation produce false negatives?
6. Why is a persistent-water pixel not necessarily deep?
7. Which errors are more dangerous for cable planning: false persistent-water detections or missed water pixels?
8. What independent data would you collect before recommending a cable alignment?

*Write your interpretation here…*

## 13. Reproducibility checklist

Before submission, confirm that you recorded:

- AOI coordinates;
- river clip polygon coordinates;
- years and seasonal window;
- STAC collection and endpoint;
- scene-level cloud threshold;
- selected scene IDs;
- excluded SCL classes;
- NDWI and NIR thresholds;
- minimum observation count;
- software environment;
- exported file names;
- known limitations.

A defensible result should make it possible for another analyst to repeat the search and obtain a comparable map.


## ✅ Summary

You have built an end-to-end cloud-native remote-sensing workflow:

1. queried Sentinel-2 through STAC;
2. selected one low-cloud image per year for 2022-2025;
3. loaded repeated observations into a geospatial data cube;
4. clipped the analysis to a river-corridor polygon;
5. masked low-quality pixels;
6. calculated NDWI;
7. estimated water probability from valid observations;
8. examined annual variability;
9. exported GIS-ready products;
10. derived candidate boat-survey zones;
11. separated a remote-sensing screening product from a true bathymetric engineering assessment.

The central lesson is that **repeated satellite observations can focus field data collection**, but safe river-cable design still depends on direct measurements of depth, substrate, flow, navigation, and hazards.
